# DASH-Q on HuggingFace Models

This notebook demonstrates how to apply the **DASH-Q** quantization algorithm to a real HuggingFace model (e.g., `Qwen/Qwen2.5-0.5B`) using a calibration dataset like `wikitext`.

The typical Post-Training Quantization (PTQ) pipeline involves:
1. Loading the model and tokenizer.
2. Running calibration data to capture activations (which are used to compute the diagonal Hessian).
3. Applying DASH-Q layer-by-layer.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

sys.path.append(os.path.abspath('src/'))
from dashq.dash_q import quantize_layer

sns.set_theme(style="whitegrid")
device = "cuda" if torch.cuda.is_available() else "cpu"

## 1. Load Model and Calibration Dataset
We'll load a small model and use `wikitext` for calibration.

In [ ]:
model_id = "Qwen/Qwen2.5-0.5B"
dataset_name = "wikitext"
dataset_config = "wikitext-2-raw-v1"

print(f"Loading tokenizer and model: {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")
model.eval()

In [ ]:
print(f"Loading {dataset_name} dataset...")
dataset = load_dataset(dataset_name, dataset_config, split="train")
text = "\n\n".join(dataset["text"])
tokens = tokenizer(text, return_tensors="pt").input_ids[0]

n_samples = 4
seq_len = 512
input_ids = tokens[:n_samples * seq_len].view(n_samples, seq_len)
print(f"Calibration data shape: {input_ids.shape}")

## 2. Capture Activations
To compute the diagonal Hessian for a specific linear layer, we need its input activations. We'll use a PyTorch forward hook on the first self-attention projection layer.

In [ ]:
# Let's select the first attention Q-projection layer
target_layer_name = "model.layers.0.self_attn.q_proj"
target_module = dict(model.named_modules())[target_layer_name]

layer_inputs = []
def hook(mod, inp, out):
    # Capture the input tensor
    layer_inputs.append(inp[0].detach().cpu())

handle = target_module.register_forward_hook(hook)

# Run the calibration data through the model
with torch.no_grad():
    for i in range(n_samples):
        batch = input_ids[i:i+1].to(device)
        model(batch)

handle.remove()

# Concatenate and reshape to (N, in_features)
X = torch.cat(layer_inputs, dim=0).view(-1, target_module.in_features).to(torch.float32)
print(f"Captured activations shape for {target_layer_name}: {X.shape}")

## 3. Apply DASH-Q
Now we pass the weights and the captured activations to `quantize_layer`. We'll target 4-bit quantization with a group size of 32 (compatible with `Q4_1`).

In [ ]:
W = target_module.weight.data.clone().to(torch.float32)

print("Running DASH-Q...")
Q, S, Z = quantize_layer(W, X, b=4, group_size=32, T=9)

# Reconstruct the weights to measure error
W_hat = torch.zeros_like(W)
num_groups = W.shape[1] // 32
for g in range(num_groups):
    start = g * 32
    end = start + 32
    W_hat[:, start:end] = S[:, g:g+1] * Q[:, start:end] - Z[:, g:g+1]

mse = torch.nn.functional.mse_loss(W_hat, W)
print(f"Original Weight Mean: {W.mean().item():.4f}, Std: {W.std().item():.4f}")
print(f"Reconstructed Weight MSE: {mse.item():.6f}")

## 4. Visualizing the Result
We can plot the weight distributions to see how well DASH-Q preserved the structure.

In [ ]:
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(W.flatten().numpy(), bins=100, alpha=0.7, label='Original (FP16)')
plt.title('Original Weights')

plt.subplot(1, 2, 2)
plt.hist(W_hat.flatten().numpy(), bins=100, alpha=0.7, color='orange', label='Quantized (4-bit)')
plt.title('DASH-Q Reconstructed Weights')

plt.tight_layout()
plt.show()